# Fase 3 — Data Preparation | CardioRisk · CRISP-DM
Preprocesamiento canónico: feature engineering, OHE, splits, SMOTE.
Exporta artefactos para F4/F5/F6 vía %store.

In [ ]:
# BLOQUE 1 — INSTALACIONES + CANON ÚNICO DE PREPROCESAMIENTO
!pip install -q kagglehub imbalanced-learn xgboost shap

import kagglehub, os, warnings, joblib
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import matplotlib, matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'figure.facecolor': '#0a0f1a', 'axes.facecolor': '#0d1526',
    'axes.edgecolor': '#1a2c3d',   'axes.labelcolor': '#e2e8f0',
    'text.color': '#e2e8f0',       'xtick.color': '#7a8fa8',
    'ytick.color': '#7a8fa8',      'grid.color': '#1a2c3d',
    'savefig.facecolor': '#0a0f1a'
})

# ── CARGA ──
try:
    path = kagglehub.dataset_download('jocelyndumlao/cardiovascular-disease-dataset')
    csv_path = next(os.path.join(r,f) for r,_,fs in os.walk(path) for f in fs if f.endswith('.csv'))
    df = pd.read_csv(csv_path)
except:
    df = pd.read_csv('/content/cardiovascular_disease_dataset.csv')

df.columns = df.columns.str.lower().str.strip()
df = df.dropna()
print(f'Dataset: {df.shape[0]} registros, {df.shape[1]} columnas')

# ── FEATURE ENGINEERING (2 features) ──
# slope_x_oldpeak: interacción clínica entre pendiente ST y depresión ST
# log_oldpeak: normaliza la distribución sesgada de oldpeak
df['slope_x_oldpeak'] = df['slope'] * df['oldpeak']
df['log_oldpeak']     = np.log1p(df['oldpeak'])
# NOTA: vessels_bin fue eliminada — no es una decisión de dominio clínico consciente

# ── OHE DEFINITIVO — drop_first=False, preserva todas las categorías clínicas ──
# Razón: gender(0/1), chestpain(0-3), restingrelectro(0-2) son categorías nominales
# drop_first=False mantiene interpretabilidad clínica de cada categoría
CATEGORICAL = ['gender', 'chestpain', 'restingrelectro']
TARGET = 'target'
df_enc = pd.get_dummies(df, columns=CATEGORICAL, drop_first=False)

# ── FEATURES FINALES — 20 features ──
# 9 numéricas + 2 engineered + 9 OHE (2+4+3) = 20
EXCLUIR = [TARGET, 'patientid']
FEATURES_FINALES = [c for c in df_enc.columns if c not in EXCLUIR]
print(f'Features: {len(FEATURES_FINALES)}')  # debe imprimir 20
print(f'Lista: {FEATURES_FINALES}')

X = df_enc[FEATURES_FINALES]
y = df_enc[TARGET]

# ── SPLITS 70/15/15 estratificado ──
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42, stratify=y_temp)

# ── SCALING — fit SOLO en train ──
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_val_sc   = scaler.transform(X_val)
X_test_sc  = scaler.transform(X_test)

# ── SMOTE — SOLO en train ──
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train_sc, y_train)

print(f'Train: {X_train_sm.shape[0]} | Val: {X_val_sc.shape[0]} | Test: {X_test_sc.shape[0]}')
print(f'Clases train tras SMOTE: {dict(zip(*np.unique(y_train_sm, return_counts=True)))}')


In [ ]:
# BLOQUE 2 — ANÁLISIS DEL FEATURE ENGINEERING
print('=== Estadísticas de features engineered ===')
print(f"slope_x_oldpeak: mean={df['slope_x_oldpeak'].mean():.3f}, std={df['slope_x_oldpeak'].std():.3f}, min={df['slope_x_oldpeak'].min():.3f}, max={df['slope_x_oldpeak'].max():.3f}")
print(f"log_oldpeak:     mean={df['log_oldpeak'].mean():.3f}, std={df['log_oldpeak'].std():.3f}, min={df['log_oldpeak'].min():.3f}, max={df['log_oldpeak'].max():.3f}")

print('\n=== Las 20 FEATURES FINALES ===')
ohe_cols = [c for c in FEATURES_FINALES if any(c.startswith(cat+'_') for cat in CATEGORICAL)]
eng_cols = ['slope_x_oldpeak','log_oldpeak']
num_cols = [c for c in FEATURES_FINALES if c not in ohe_cols and c not in eng_cols]
print(f'Numéricas ({len(num_cols)}): {num_cols}')
print(f'Engineered ({len(eng_cols)}): {eng_cols}')
print(f'OHE ({len(ohe_cols)}): {ohe_cols}')
print(f'TOTAL: {len(FEATURES_FINALES)}')


In [ ]:
# BLOQUE 3 — BALANCE DE CLASES ANTES/DESPUÉS DE SMOTE
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

counts_before = y_train.value_counts().sort_index()
bars1 = ax1.bar(['Bajo Riesgo (0)', 'Alto Riesgo (1)'], counts_before.values, color=['#3b82f6','#ef4444'], alpha=0.8)
ax1.set_title('Antes de SMOTE', fontsize=13, fontweight='bold')
ax1.set_ylabel('Cantidad')
for bar in bars1:
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, str(int(bar.get_height())), ha='center', fontweight='bold')

counts_after = pd.Series(y_train_sm).value_counts().sort_index()
bars2 = ax2.bar(['Bajo Riesgo (0)', 'Alto Riesgo (1)'], counts_after.values, color=['#3b82f6','#ef4444'], alpha=0.8)
ax2.set_title('Después de SMOTE', fontsize=13, fontweight='bold')
ax2.set_ylabel('Cantidad')
for bar in bars2:
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+2, str(int(bar.get_height())), ha='center', fontweight='bold')

plt.suptitle('Balance de Clases — Train Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/f3_class_balance.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# BLOQUE 4 — VERIFICACIÓN DE NO DATA LEAKAGE
print('=== Verificación de Data Leakage ===')
print(f'Scaler fit en X_train ({X_train.shape[0]} muestras): {scaler.n_samples_seen_ == X_train.shape[0]}')
print(f'X_train_sm shape (SMOTE solo en train): {X_train_sm.shape}')
print(f'X_val_sc   shape (sin SMOTE): {X_val_sc.shape}')
print(f'X_test_sc  shape (sin SMOTE): {X_test_sc.shape}')
print(f'Val  clases: {dict(zip(*np.unique(y_val,  return_counts=True)))}')
print(f'Test clases: {dict(zip(*np.unique(y_test, return_counts=True)))}')
print('✓ Sin data leakage confirmado')


In [ ]:
# BLOQUE 5 — EXPORT DE ARTEFACTOS
%store X_train_sm
%store y_train_sm
%store X_val_sc
%store y_val
%store X_test_sc
%store y_test
%store scaler
%store FEATURES_FINALES
print('✓ F3 COMPLETO — 20 features exportadas vía %store')
